# 15 SOTA Reward Comparison

Run this notebook after the previous numbered stage completes.


In [1]:
from pathlib import Path
import os, sys

REPO_ROOT_OVERRIDE = None


def _looks_like_repo(base: Path) -> bool:
    return (base / "pyproject.toml").exists() and (base / "src" / "cybsage_marl").exists()


def _mount_drive_if_colab():
    if not Path("/content").exists():
        return
    if Path("/content/drive/MyDrive").exists() or Path("/content/drive/My Drive").exists():
        return
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception as exc:
        print("Drive auto-mount failed:", exc)


def _find_repo_root():
    _mount_drive_if_colab()
    candidates = []
    if REPO_ROOT_OVERRIDE:
        candidates.append(Path(REPO_ROOT_OVERRIDE).expanduser())
    if os.environ.get("CYBSAGE_REPO_ROOT"):
        candidates.append(Path(os.environ["CYBSAGE_REPO_ROOT"]).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.extend([
        Path("/content/drive/MyDrive/projects/implementation/IMPLEMENTATION"),
        Path("/content/drive/My Drive/projects/implementation/IMPLEMENTATION"),
        Path("/content/drive/MyDrive/IMPLEMENTATION"),
        Path("/content/drive/My Drive/IMPLEMENTATION"),
        Path("/content/drive/MyDrive/projects/implementation/CybSage-MARL[DeepakSir]/IMPLEMENTATION"),
        Path("/content/drive/My Drive/projects/implementation/CybSage-MARL[DeepakSir]/IMPLEMENTATION"),
        Path("/Volumes/LocalDrive1/onlyscholar-projects/CybSage-MARL[DeepakSir]/IMPLEMENTATION"),
    ])

    checked = []
    seen = set()
    for base in candidates:
        base = base.expanduser()
        if str(base) in seen:
            continue
        seen.add(str(base))
        checked.append(base)
        if _looks_like_repo(base):
            return base

    # Last-resort bounded search. This catches Drive shortcuts/copies while avoiding a full Drive crawl.
    search_roots = [Path("/content/drive/MyDrive"), Path("/content/drive/My Drive"), Path("/content")]
    for root in search_roots:
        if not root.exists():
            continue
        try:
            for pyproject in root.glob("**/pyproject.toml"):
                base = pyproject.parent
                checked.append(base)
                if base.name == "IMPLEMENTATION" and _looks_like_repo(base):
                    return base
        except Exception as exc:
            checked.append(Path(f"{root} [search failed: {exc}]"))

    raise RuntimeError(
        "Could not find CybSAGE IMPLEMENTATION repo root visible to this runtime.\n\n"
        "Most common fix: in Colab, run drive.mount('/content/drive') and confirm the folder exists under "
        "/content/drive/MyDrive/projects/implementation/IMPLEMENTATION.\n\n"
        "Checked:\n" + "\n".join(map(str, checked[-80:]))
    )


REPO_ROOT = _find_repo_root()
os.chdir(REPO_ROOT)
src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print("Repository root:", REPO_ROOT)


Repository root: /content/drive/MyDrive/projects/implementation/IMPLEMENTATION


In [2]:
!PYTHONPATH=src python scripts/run_stage.py cage4_sota

CAGE-4 SOTA comparison summary: {'status': 'completed', 'baselines_compared': 6, 'superiority_claim_ready': False, 'claim_boundary': 'Do not claim SOTA superiority yet; at least one baseline was not beaten with positive confidence.'}


In [3]:
import json, pandas as pd
from pathlib import Path
summary = json.loads(Path("outputs/15_sota_reward_comparison/cage4_sota_summary.json").read_text())
print(summary)
comparison = Path("outputs/15_sota_reward_comparison/cage4_sota_comparison.csv")
pd.read_csv(comparison) if comparison.exists() and comparison.stat().st_size else "No comparison table produced"

{'status': 'completed', 'baselines_compared': 6, 'superiority_claim_ready': False, 'claim_boundary': 'Do not claim SOTA superiority yet; at least one baseline was not beaten with positive confidence.'}


,baseline,paired_runs,cybsage_mean,baseline_mean,mean_difference,ci95_low,ci95_high,sign_test_p_value,cybsage_wins
0,restore_priority,6,-63.5,-552.500000,489.000000,457.000000,521.000000,0.03125,6
1,random,6,-63.5,-503.833333,440.333333,301.833333,557.166667,0.03125,6
2,sleep,6,-63.5,-82.166667,18.666667,1.833333,38.333333,0.21875,5
3,monitor,6,-63.5,-82.166667,18.666667,1.833333,38.333333,0.21875,5
4,remove_priority,6,-63.5,-72.166667,8.666667,-4.000000,21.333333,0.21875,5
5,decoy_priority,6,-63.5,-63.500000,0.000000,0.000000,0.000000,1.00000,0


In [4]:
from pathlib import Path
report = Path("outputs/15_sota_reward_comparison/cage4_sota_report.md")
print(report.read_text()[:4000] if report.exists() else "No report produced")

# CAGE-4 Reward SOTA Comparison

- `status`: `completed`
- `superiority_claim_ready`: `False`
- `claim_boundary`: Do not claim SOTA superiority yet; at least one baseline was not beaten with positive confidence.

## Baseline Comparisons

| baseline | cybsage_mean | baseline_mean | mean_difference | ci95_low | ci95_high | sign_test_p_value | cybsage_wins | paired_runs |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| restore_priority | -63.5 | -552.5 | 489 | 457 | 521 | 0.03125 | 6 | 6 |
| random | -63.5 | -503.833 | 440.333 | 301.833 | 557.167 | 0.03125 | 6 | 6 |
| sleep | -63.5 | -82.1667 | 18.6667 | 1.83333 | 38.3333 | 0.21875 | 5 | 6 |
| monitor | -63.5 | -82.1667 | 18.6667 | 1.83333 | 38.3333 | 0.21875 | 5 | 6 |
| remove_priority | -63.5 | -72.1667 | 8.66667 | -4 | 21.3333 | 0.21875 | 5 | 6 |
| decoy_priority | -63.5 | -63.5 | 0 | 0 | 0 | 1 | 0 | 6 |

